Problem 1: Linear Regression (ordinary least squares)
                          
                             (i)  Ridge Regression

                             (ii) Lasso Regression  (Coordinate Descent, Soft Thresholding Operator)

In [32]:
import numpy as np
import matplotlib.pyplot as plt

class RidgeRegression():
    def __init__(self,lambd=1.0, method='solve'):
        self.method = method
        self.lambd = lambd


    def fit(self, X, Y):
        N, D = X.shape  # number of samples and features
        if self.method == 'solve':
            matrix1 = X.T@X + self.lambd*np.eye(D)
            return np.linalg.solve(matrix1, X.T@Y)
        elif self.method == 'svd':
            U, S, VT = np.linalg.svd(X, full_matrices=False)
            d_factor = S/(S**2 + self.lambd)
            print(f"U shape: {U.shape}, S shape: {S.shape}, VT shape: {VT.shape}, d_factor shape: {d_factor.shape}")
            UT_Y = U.T @ Y
            if UT_Y.ndim > 1:
                d_factor = d_factor[:, None]
                
            return VT.T @ (d_factor * UT_Y)

    def predict(self, X, Beta):
        return X@Beta

class LassoRegression():
    def __init__(self, lambd=1.0, tol=1e-5, max_iter=10000):
        self.lambd = lambd
        self.tol = tol
        self.max_iter = max_iter

    def fit(self, X, Y):

        N, D = X.shape  # number of samples and features

        beta = np.zeros(D)

        residual = Y - X@beta

        sample_nrmlized_featr_val  = (1/N) * np.sum(X**2, axis=0)

        for iteration in range(self.max_iter):
            old_beta = beta.copy()
            for k in range(D):
                if sample_nrmlized_featr_val[k] == 0:
                    continue

                X_k = X[:,k]
                rho_k = (1/N)*np.dot(X_k, residual)+ beta[k]*sample_nrmlized_featr_val[k] 

                if rho_k>= self.lambd:
                    beta_new = (rho_k-self.lambd)/sample_nrmlized_featr_val[k]
                elif rho_k<=-self.lambd:
                    beta_new = (rho_k + self.lambd)/sample_nrmlized_featr_val[k]

                else:
                    beta_new = 0

                if beta_new != beta[k]:
                    residual += X_k*(beta[k]-beta_new)
                    beta[k] = beta_new
     

            if np.max(np.abs(old_beta-beta)) < self.tol:
                break


        return beta

    
    def predict(self, X, beta):
        return X@beta


def data_generator(N=1000, D=10, reduced_dim = 5,noise_std=0.1):
    np.random.seed(0)
    X = np.random.randn(N, D)
    true_beta = np.zeros(D)
    true_beta[:reduced_dim] = np.random.randn(reduced_dim)
    Y = X @ true_beta + noise_std * np.random.randn(N)
    return X, Y, true_beta


X, Y, true_beta = data_generator()


lasso = LassoRegression(lambd=0.02)
beta = lasso.fit(X, Y)
predicted_y = lasso.predict(X, beta)

print(f"beta:\n{np.round(beta, 4)}")
print(f"true_beta:\n{np.round(true_beta, 4)}")
print("Mean Squared Error:", np.mean((predicted_y - Y)**2))



beta = RidgeRegression(method='solve').fit(X,Y)
predicted_y = RidgeRegression().predict(X,beta)

print(f"beta: {beta}")
print(f"true_beta: {true_beta}")
Error = np.mean((predicted_y - Y)**2)
print("Mean Squared Error:", Error)




beta = RidgeRegression(method='svd').fit(X,Y)
predicted_y = RidgeRegression().predict(X,beta)

print(f"beta: {beta}")
print(f"true_beta: {true_beta}")
Error = np.mean((predicted_y - Y)**2)
print("Mean Squared Error:", Error)


beta:
[-0.1826 -0.816   1.7186  0.1702 -0.1543  0.      0.      0.      0.
  0.    ]
true_beta:
[-0.2021 -0.8332  1.7336  0.1906 -0.1778  0.      0.      0.      0.
  0.    ]
Mean Squared Error: 0.012053391435646762
beta: [-2.03013414e-01 -8.35226738e-01  1.73639533e+00  1.92700559e-01
 -1.74934583e-01  9.77121165e-04  3.47595604e-04  2.09118414e-03
 -9.52685227e-04  4.72607033e-03]
true_beta: [-0.20211703 -0.833231    1.73360025  0.190649   -0.17781039  0.
  0.          0.          0.          0.        ]
Mean Squared Error: 0.009959680344770074
U shape: (1000, 10), S shape: (10,), VT shape: (10, 10), d_factor shape: (10,)
beta: [-2.03013414e-01 -8.35226738e-01  1.73639533e+00  1.92700559e-01
 -1.74934583e-01  9.77121165e-04  3.47595604e-04  2.09118414e-03
 -9.52685227e-04  4.72607033e-03]
true_beta: [-0.20211703 -0.833231    1.73360025  0.190649   -0.17781039  0.
  0.          0.          0.          0.        ]
Mean Squared Error: 0.00995968034477007


Problem 2: Implement PCA using both the Sample Covariance Matrix (Eigendecomposition) and SVD, then extend it to Kernel PCA (RBF kernel) for non-linear dimensionality reduction.

In [20]:
import numpy as np
import matplotlib.pyplot as plt

class pca:
    def __init__(self, no_of_components=15):
        self.no_of_components= no_of_components
        self.variance_=None
        self.PCs_ = None
        self.mean_ = None

    def fit(self, X):
        N, D = X.shape # no. of datapoints and features
        self.mean_ = np.mean(X, axis=0)

        X_centered = X - self.mean_[None, :]

        U, S, VT= np.linalg.svd(X_centered, full_matrices=False)

        self.PCs_ = VT[:self.no_of_components]

        self.variance_ = ((S**2) / (N - 1)) [:self.no_of_components]
        

        return self

    def transform(self, X):
        X_centered = X - self.mean_[None, :]
        return X_centered@(self.PCs_.T)


class kernel_pca:
    def __init__(self, no_of_components = 15, gamma = 1.0):
        self.no_of_components = no_of_components
        self.gamma = gamma
        self.K_centered_ = None
        self.alphas_ = None
        self.lambdas_ = None
        self.X_fit_ = None
        self.K_fit_rows_mean_ = None
        self.K_fit_all_mean_ = None


    def rbf_kernel(self, X1, X2):

        X1_squar = np.sum(X1**2, axis = 1)[:, None]
        X2_squar = np.sum(X2**2, axis = 1)[None, :]

        dist = X1_squar + X2_squar - 2*(X1@(X2.T))
        return np.exp(-self.gamma*dist)

    def fit(self, X):
        self.X_fit_ = X
        N, D = X.shape
        kernel = self.rbf_kernel(X, X)
        one_mat = np.ones((N,N))/N

        self.K_fit_rows_mean_ = np.mean(kernel, axis=0, keepdims=True)  # (1, N)
        self.K_fit_all_mean_ = np.mean(kernel)

        self.K_centered_ =  kernel - one_mat@kernel - kernel@one_mat + one_mat@(kernel@one_mat)     # Double Centering 

        eigvals, eigvecs = np.linalg.eigh(self.K_centered_)
        idx= np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        positive_idx = eigvals > 1e-10
        self.lambdas_ = eigvals[positive_idx][:self.no_of_components]
        self.alphas_ = eigvecs[:, positive_idx][:, :self.no_of_components]

        self.alphas_ = self.alphas_ / np.sqrt(self.lambdas_)
        return self

    def transform(self, X_test):
        K_test = self.rbf_kernel(X_test, self.X_fit_)  # (M, N)
        
        # Center test kernel using training kernel statistics
        K_test_rows_mean = np.mean(K_test, axis=1, keepdims=True)  # (M, 1)
        K_test_centered = K_test - self.K_fit_rows_mean_ - K_test_rows_mean + self.K_fit_all_mean_

        # Project centered kernel onto normalized eigenvectors
        return K_test_centered @ self.alphas_






rng = np.random.default_rng(seed=42)
X = rng.normal(loc = 0.0, scale = 1.0, size = (500, 50))
X_test = rng.normal(loc = 0.0, scale = 1.0, size = (100, 50))

pca_model = pca(no_of_components=15).fit(X)
Z_test_pca = pca_model.transform(X_test)
print("Linear PCA projected test shape:", Z_test_pca.shape)

# Kernel PCA
kpca_model = kernel_pca(no_of_components=15, gamma=0.01).fit(X)
Z_test_kpca = kpca_model.transform(X_test)
print("Kernel PCA projected test shape:", Z_test_kpca.shape)

Linear PCA projected test shape: (100, 15)
Kernel PCA projected test shape: (100, 15)


# Logistic Regression